# Kev 9B Dataset Collection (Base vs. Fine-Tuned, Default CPU)

In [1]:
# Collect data for different snapshots of Kev (trained or baseline)
# same examples as the ones for gemini models
# start with Base Kev 9B and "kev-9b-resp-issp+twin2k-grouped-newvalid"
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import wang_mnl
import kev_eval

importlib.reload(wang_mnl)
importlib.reload(kev_eval)
from wang_mnl import load_covid_conjoint, pair_choice_sets
from kev_eval import (
    build_kev_prompt_items,
    build_respondent_states,
    collect_kev_prompts,
    free_kev_memory,
    load_kev_model,
    preview_kev_item,
    verify_matches_gemini_subsample,
)

DATA_PATH = Path("../../data/covid/dataverse_files/Kreps_etal_vax_replication_data.dta")
raw_df = load_covid_conjoint(DATA_PATH)
df_A, df_B, _, _, _ = pair_choice_sets(raw_df)

# Subsample the exact same 200 respondents (1,000 choice sets, seed=42) used for Gemini
N_LLM_RESPONDENTS = 200
SUBSAMPLE_SEED = 42
all_respondents = np.sort(df_A["respondent"].unique())
sampled_respondents = np.sort(np.random.default_rng(SUBSAMPLE_SEED).choice(all_respondents, size=N_LLM_RESPONDENTS, replace=False))

sub_mask = df_A["respondent"].isin(sampled_respondents).to_numpy()
sub_df_A = df_A.loc[sub_mask].reset_index(drop=True)
sub_df_B = df_B.loc[sub_mask].reset_index(drop=True)

# Build Kev persona states and prompt items for Set 1 (Random Persona) and Set 2 (Actual Respondent Persona)
resp_state_by_id = build_respondent_states(df_A, include_country=False)
kev_random_items = build_kev_prompt_items(sub_df_A, sub_df_B, resp_state_by_id, persona_mode="random")
kev_persona_items = build_kev_prompt_items(sub_df_A, sub_df_B, resp_state_by_id, persona_mode="persona")

# Fail fast if the 200-respondent (1,000-pair) subsample does not exactly match the Gemini dataset
verify_matches_gemini_subsample(kev_random_items)
verify_matches_gemini_subsample(kev_persona_items)

print(f"Verified exact match with Gemini subsample: {len(kev_random_items):,} pairs across {len(sampled_respondents)} respondents.")

Verified exact match with Gemini subsample: 1,000 pairs across 200 respondents.


In [2]:
len(kev_persona_items)
kev_persona_items[0]

{'respondent': 14,
 'pair_id': 1,
 'persona_mode': 'persona',
 'state': 'Sex: Female\nAge: 18-29\nEducation: High school or some college\nReligion: Nothing in particular\nPolitical ideology: Moderate\nPolitical party: Something else\nIncome: $20,000 - $39,999\nRace: Black',
 'question_spec': {'type': 'choice',
  'instructions': 'Suppose you are deciding whether or not to receive a COVID-19 vaccination. There are three options for you. Which option would you choose?',
  'criteria': {'A': 'Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 10. Authorization: Approved and licensed by the US Food and Drug Administration. Origin: United Kingdom. Endorsement: World Health Organization.',
   'B': 'Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 30. Authorization: Received an emergency use authorization from the US Food and Drug Administration.  This allows the expedited use of promising drug

In [3]:
# Step 2: Preview the exact tokenized Kev format for Sampled Pair 0 (Random vs. Actual Respondent Persona)
print("=== 1. KEV RANDOM PERSONA SAMPLE (Sampled Pair 0) ===")
print(preview_kev_item(kev_random_items[0]))
print("\n" + "=" * 70 + "\n")
print("=== 2. KEV ACTUAL RESPONDENT PERSONA SAMPLE (Sampled Pair 0) ===")
print(preview_kev_item(kev_persona_items[0]))

=== 1. KEV RANDOM PERSONA SAMPLE (Sampled Pair 0) ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

217 tokens; state 1 tokens (no truncation, strict=True)
<|fim_prefix|>
<|fim_middle|>
Suppose you are deciding whether or not to receive a COVID-19 vaccination. There are three options for you. Which option would you choose?
<|box_start|>A: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 10. Authorization: Approved and licensed by the US Food and Drug Administration. Origin: United Kingdom. Endorsement: World Health Organization.<|box_end|>
<|box_start|>B: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 30. Authorization: Received an emergency use authorization from the US Food and Drug Administration.  This allows the expedited use of promising drugs that the FDA has found it reasonable to believe may be effective in combatting the virus. Origin: United Kingdom. Endorsement: Vice President Joe Biden.<|box_end|>
<|box_start|>C: You choose neither A nor B.<|box_end|>
<|fim_suffix|>


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

277 tokens; state 61 tokens (no truncation, strict=True)
<|fim_prefix|>
Sex: Female
Age: 18-29
Education: High school or some college
Religion: Nothing in particular
Political ideology: Moderate
Political party: Something else
Income: $20,000 - $39,999
Race: Black
<|fim_middle|>
Suppose you are deciding whether or not to receive a COVID-19 vaccination. There are three options for you. Which option would you choose?
<|box_start|>A: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 10. Authorization: Approved and licensed by the US Food and Drug Administration. Origin: United Kingdom. Endorsement: World Health Organization.<|box_end|>
<|box_start|>B: Efficacy: 50%. Protective duration: 5 years. Major side effect: 1 in 10,000. Minor side effect: 1 in 30. Authorization: Received an emergency use authorization from the US Food and Drug Administration.  This allows the expedited use of promising drugs that the FDA has found it reasonable to 

In [10]:
# Step 3: Configure device (default CPU), output directory, and number of personas to query
DEVICE = "cpu"  # Default CPU so GPU training runs are unaffected
DTYPE = torch.bfloat16
# N_QUERY_PERSONAS = 5  # Set to None (or 200) to collect all 200 respondents (1,000 choice sets)
N_QUERY_PERSONAS = None  # Set to None (or 200) to collect all 200 respondents (1,000 choice sets)

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [11]:
# Step 4: Collect Base Kev 9B ("jaredpalmer/kev-9b") on Set 1 (Random Persona) and Set 2 (Actual Respondent Persona)
importlib.reload(kev_eval)
from kev_eval import collect_kev_prompts, free_kev_memory, load_kev_model

base_ckpt = "jaredpalmer/kev-9b"
base_slug = "kev-9b-base"

# Clear any prior model reference and empty RAM / CUDA cache before loading
model = None
free_kev_memory()
model = load_kev_model(base_ckpt, device=DEVICE, dtype=DTYPE)
try:
    base_rand_records = collect_kev_prompts(
        model,
        kev_random_items,
        output_path=DATA_DIR / f"{base_slug}_random.jsonl",
        model_name=base_ckpt,
        n_personas=N_QUERY_PERSONAS,
    )
    base_pers_records = collect_kev_prompts(
        model,
        kev_persona_items,
        output_path=DATA_DIR / f"{base_slug}_persona.jsonl",
        model_name=base_ckpt,
        n_personas=N_QUERY_PERSONAS,
    )
finally:
    model = None
    free_kev_memory()

pd.DataFrame({
    "Base Kev 9B (Random) mean probs": np.mean([r["probs"] for r in base_rand_records], axis=0),
    "Base Kev 9B (Persona) mean probs": np.mean([r["probs"] for r in base_pers_records], axis=0),
}, index=["A", "B", "C"]).round(4)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[kev-9b-base_random.jsonl] model=jaredpalmer/kev-9b, persona_mode=random, temp=2.19, device=cpu | Target: 1000 prompts (25 already collected, 975 remaining)


kev-9b-base_random:   0%|          | 0/975 [00:00<?, ?pair/s]

[kev-9b-base_persona.jsonl] model=jaredpalmer/kev-9b, persona_mode=persona, temp=2.19, device=cpu | Target: 1000 prompts (25 already collected, 975 remaining)


kev-9b-base_persona:   0%|          | 0/975 [00:00<?, ?pair/s]

,Base Kev 9B (Random) mean probs,Base Kev 9B (Persona) mean probs
A,0.3494,0.4151
B,0.3402,0.3238
C,0.3104,0.2611


In [12]:
# Step 5: Collect Fine-Tuned Kev 9B ("kev-9b-resp-issp+twin2k-grouped-newvalid") on Set 1 & Set 2
# Choose snapshot = "best" or "last"
importlib.reload(kev_eval)
from kev_eval import collect_kev_prompts, free_kev_memory, load_kev_model

ft_run = "kev-9b-resp-issp+twin2k-grouped-newvalid"
ft_snapshot = "best"
ft_ckpt = f"../../runs/{ft_run}/{ft_snapshot}"
ft_slug = f"{ft_run}_{ft_snapshot}"

# Clear any prior model reference and empty RAM / CUDA cache before loading
model = None
free_kev_memory()
model = load_kev_model(ft_ckpt, device=DEVICE, dtype=DTYPE)
try:
    ft_rand_records = collect_kev_prompts(
        model,
        kev_random_items,
        output_path=DATA_DIR / f"{ft_slug}_random.jsonl",
        model_name=f"{ft_run}/{ft_snapshot}",
        n_personas=N_QUERY_PERSONAS,
    )
    ft_pers_records = collect_kev_prompts(
        model,
        kev_persona_items,
        output_path=DATA_DIR / f"{ft_slug}_persona.jsonl",
        model_name=f"{ft_run}/{ft_snapshot}",
        n_personas=N_QUERY_PERSONAS,
    )
finally:
    model = None
    free_kev_memory()

pd.DataFrame({
    f"{ft_slug} (Random) mean probs": np.mean([r["probs"] for r in ft_rand_records], axis=0),
    f"{ft_slug} (Persona) mean probs": np.mean([r["probs"] for r in ft_pers_records], axis=0),
}, index=["A", "B", "C"]).round(4)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

[kev-9b-resp-issp+twin2k-grouped-newvalid_best_random.jsonl] model=kev-9b-resp-issp+twin2k-grouped-newvalid/best, persona_mode=random, temp=1.00, device=cpu | Target: 1000 prompts (25 already collected, 975 remaining)


kev-9b-resp-issp+twin2k-grouped-newvalid_best_random:   0%|          | 0/975 [00:00<?, ?pair/s]

[kev-9b-resp-issp+twin2k-grouped-newvalid_best_persona.jsonl] model=kev-9b-resp-issp+twin2k-grouped-newvalid/best, persona_mode=persona, temp=1.00, device=cpu | Target: 1000 prompts (25 already collected, 975 remaining)


kev-9b-resp-issp+twin2k-grouped-newvalid_best_persona:   0%|          | 0/975 [00:00<?, ?pair/s]

,kev-9b-resp-issp+twin2k-grouped-newvalid_best (Random) mean probs,kev-9b-resp-issp+twin2k-grouped-newvalid_best (Persona) mean probs
A,0.4123,0.3671
B,0.3393,0.3105
C,0.2484,0.3224
